# Klasifikasi DemogPairs Menggunakan ViT (Wajah) & SVM

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
features = joblib.load('features/demogpairs_vit-face.pkl')
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

Jumlah fitur per gambar: 768


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
from sklearn.decomposition import PCA

grid_params = [
    {
        'scaler': [None, MinMaxScaler()],  # EN: scaling option / ID: opsi scaling
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],  # EN: with and without PCA / ID: dengan dan tanpa PCA
        
        'classifier': [SVC()],  # EN: SVM model / ID: model SVM
        
        'classifier__C': [0.01, 0.1, 1, 10],  # EN: regularization / ID: regularisasi
        'classifier__kernel': ['rbf', 'poly', 'linear'],  # EN: kernel types / ID: jenis kernel
        
        'classifier__gamma': ['scale', 'auto'],  # EN: gamma / ID: gamma
        'classifier__degree': [2, 3],  # EN: degree / ID: derajat
        
        'classifier__tol': [1e-3],  # EN: tolerance / ID: toleransi
        'classifier__probability': [True],  # EN: probability / ID: probabilitas
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),  # EN: optional scaler / ID: scaler opsional
    ('pca', None),     # EN: optional PCA / ID: PCA opsional
    ('classifier', None)  # EN: classifier / ID: classifier
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro'
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

SVC: 288 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models,
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix='models/clf_demogpairs_svm_vit-face_',
    results_path='results/demogpairs_svm_vit-face_'
)

sorted_results = pd.DataFrame(evaluation_results).sort_values(by='test_accuracy', ascending=False).to_dict('records')
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: SVC


{'classifier': 'SVC', 'classifier__C': 10, 'classifier__degree': 2, 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf', 'classifier__probability': True, 'classifier__tol': 0.001, 'pca': None, 'scaler': None}


Accuracy  : 0.9083333333333333
Precision : 0.9083778499800578
Recall    : 0.9083333333333332
F1 Score  : 0.9082783206930735
               precision    recall  f1-score   support

Asian_Females     0.9407    0.9250    0.9328       360
  Asian_Males     0.8978    0.9028    0.9003       360
Black_Females     0.8919    0.9167    0.9041       360
  Black_Males     0.9290    0.9444    0.9366       360
White_Females     0.8964    0.8889    0.8926       360
  White_Males     0.8946    0.8722    0.8833       360

     accuracy                         0.9083      2160
    macro avg     0.9084    0.9083    0.9083      2160
 weighted avg     0.9084    0.9083    0.9083      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9777777777777777,0.940677966101695,0.925,0.9327731092436976,360
Asian_Males,0.9666666666666667,0.8977900552486188,0.9027777777777778,0.9002770083102494,360
Black_Females,0.9675925925925926,0.8918918918918919,0.9166666666666666,0.9041095890410958,360
Black_Males,0.9787037037037037,0.9289617486338798,0.9444444444444444,0.9366391184573003,360
White_Females,0.9643518518518519,0.896358543417367,0.8888888888888888,0.8926080892608089,360
White_Males,0.961574074074074,0.8945868945868946,0.8722222222222222,0.8832630098452883,360


Confusion matrix saved: images\cm_svm_vit-face_SVC.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               333                 1                10                 6                10                 0
         Asian_Males                 0               325                 2                 5                 8                20
       Black_Females                 6                 1               330                12                 3                 8
         Black_Males                 6                 4                10               340                 0                 0
       White_Females                 9                15                 6                 1               320                 9
         White_Males                 0                16                12                 2                16               314


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
SVC,models/clf_demogpairs_svm_vit-face_SVC.pkl,"{'classifier': 'SVC', 'classifier__C': 10, 'classifier__degree': 2, 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf', 'classifier__probability': True, 'classifier__tol': 0.001, 'pca': None, 'scaler': None}",0.9083333333333333,0.9082783206930735,0.9083778499800578,0.9083333333333332,288


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_svm_vit-face_SVC.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 10277.0,
 'days': 0,
 'hours': 2,
 'minutes': 51,
 'seconds': 17.0,
 'text': '0 hari 2 jam 51 menit 17.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 83703.0,
 'days': 0,
 'hours': 23,
 'minutes': 15,
 'seconds': 3.0,
 'text': '0 hari 23 jam 15 menit 3.0 detik'}

In [10]:
def remove_keys(data, keys_to_remove, inplace=False):
    # EN: if not inplace, create a copy / ID: jika tidak inplace, buat salinan
    if not inplace:
        data = data.copy()
    
    # EN: loop through keys and remove safely / ID: loop key dan hapus dengan aman
    for key in keys_to_remove:
        data.pop(key, None)  # EN: avoid error if key not found / ID: aman jika key tidak ada
    
    return data

def dict_to_sentence(d):
    # EN: convert key-value pairs into readable parts / ID: ubah key-value jadi bagian kalimat
    parts = [f"{k}={v}" for k, v in d.items()]
    
    # EN: join all parts into one sentence / ID: gabungkan jadi satu kalimat
    sentence = ", ".join(parts)
    
    return sentence

fold_displays = []
for r in [remove_keys(r, ['No', 'F1 Score Mean', 'Precision Mean', 'Recall Mean', 'Train Time Mean']) for r in fold_results]:
    r['Params'] = dict_to_sentence(remove_keys(r['Params'], ['classifier'])).replace('classifier__', '')
    r['Mean'] = r['Accuracy Mean']
    del r['Accuracy Mean']
    fold_displays.append(r)
fold_displays = [{'No': idx + 1, **r} for idx, r in enumerate(sorted(fold_displays, key=lambda x: x['Mean'], reverse=True))]
_dtable = u.display_table(fold_displays, n_items=[3, 3, 3, 3], column_widths=['5%', '65%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Mean
1,"C=10, degree=3, gamma=scale, kernel=rbf, probability=True, tol=0.001, pca=None, scaler=None",0.9265,0.912,0.9039,0.9103,0.9178,0.9141
2,"C=10, degree=2, gamma=scale, kernel=rbf, probability=True, tol=0.001, pca=None, scaler=None",0.9265,0.912,0.9039,0.9103,0.9178,0.9141
3,"C=10, degree=2, gamma=scale, kernel=rbf, probability=True, tol=0.001, pca=None, scaler=MinMaxScaler",0.9225,0.9132,0.9039,0.9132,0.9172,0.914
...,...,...,...,...,...,...,...
96,"C=10, degree=3, gamma=auto, kernel=linear, probability=True, tol=0.001, pca=PCA, scaler=MinMaxScaler",0.8906,0.8767,0.8756,0.8796,0.8808,0.8807
97,"C=10, degree=2, gamma=auto, kernel=linear, probability=True, tol=0.001, pca=PCA, scaler=MinMaxScaler",0.8906,0.8767,0.8756,0.8796,0.8808,0.8807
98,"C=10, degree=2, gamma=auto, kernel=poly, probability=True, tol=0.001, pca=PCA, scaler=MinMaxScaler",0.8924,0.8762,0.8744,0.8825,0.8773,0.8806
...,...,...,...,...,...,...,...
191,"C=1, degree=2, gamma=auto, kernel=poly, probability=True, tol=0.001, pca=PCA, scaler=MinMaxScaler",0.8744,0.8501,0.8605,0.8553,0.864,0.8609
192,"C=0.01, degree=3, gamma=auto, kernel=poly, probability=True, tol=0.001, pca=PCA, scaler=None",0.8698,0.8588,0.8542,0.853,0.8652,0.8602
